<a href="https://colab.research.google.com/github/BetaCom16/Bachelorarbeit_ESP32S3-/blob/main/Bachelorarbeit_Colab_ESP32S3_Pardey.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import zipfile

with zipfile.ZipFile("/content/dataset.zip", "r") as zip_ref:
  zip_ref.extractall("/content/")

In [ ]:
import tensorflow as tf

bild_groesse = (96, 96)
batch_groesse = 32
datensatz_pfad = "/content/dataset"

train_ds = tf.keras.utils.image_dataset_from_directory(
    datensatz_pfad,
    validation_split = 0.2,
    subset = "training",
    seed = 48,
    image_size = bild_groesse,
    batch_size = None,
    crop_to_aspect_ratio=True,
    interpolation="nearest"
)

rest_ds = tf.keras.utils.image_dataset_from_directory(
    datensatz_pfad,
    validation_split = 0.2,
    subset = "validation",
    seed = 48,
    image_size = bild_groesse,
    batch_size = None,
    crop_to_aspect_ratio=True,
    interpolation="nearest"
)

anzahl_rest = tf.data.experimental.cardinality(rest_ds).numpy()
haelfte = anzahl_rest // 2

val_ds = rest_ds.take(haelfte)
test_ds = rest_ds.skip(haelfte)

train_ds = train_ds.batch(batch_groesse)
val_ds = val_ds.batch(batch_groesse)
test_ds = test_ds.batch(batch_groesse)

In [ ]:
basis_modell_mobilenet_v1 = tf.keras.applications.MobileNet(
    input_shape=(96, 96, 3),
    alpha=0.25,
    include_top=False,
    weights="imagenet"
)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.1),
])

basis_modell_mobilenet_v1.trainable = False

modell_mobilenet_v1 = tf.keras.Sequential(
    [
    tf.keras.layers.Input(shape=(96, 96, 3)),
    data_augmentation,
    tf.keras.layers.Lambda(tf.keras.applications.mobilenet.preprocess_input),
    basis_modell_mobilenet_v1,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(3, activation="softmax")
    ], name="mobilenet_v1"
  )

modell_mobilenet_v1.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_mobilenet_v1 = modell_mobilenet_v1.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15
)

In [ ]:
basis_modell_mobilenet_v2 = tf.keras.applications.MobileNetV2(
    input_shape=(96, 96, 3),
    alpha=0.35,
    include_top=False,
    weights="imagenet"
)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.1),
])

basis_modell_mobilenet_v2.trainable = False

modell_mobilenet_v2 = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(96, 96, 3)),
    data_augmentation,
    tf.keras.layers.Lambda(tf.keras.applications.mobilenet_v2.preprocess_input),
    basis_modell_mobilenet_v2,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(3, activation="softmax")
], name="mobilenet_v2")

modell_mobilenet_v2.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_mobilenet_v2 = modell_mobilenet_v2.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15
)

In [ ]:
import tensorflow as tf

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.1),
])

modell_tinycnn = tf.keras.Sequential([

    tf.keras.layers.Input(shape=(96, 96, 3)),
    data_augmentation,
    tf.keras.layers.Rescaling(1./255),
    tf.keras.layers.Conv2D(8, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(16, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(24, 3, activation="relu"),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(3, activation="softmax")
], name="tinycnn")

modell_tinycnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_tinycnn = modell_tinycnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15
)

In [ ]:
def representative_dataset_mobilenet_v1():
    for bilder, _ in train_ds.unbatch().batch(1).take(100):
        yield [tf.cast(bilder, tf.float32)]

converter_mobilenet_v1 = tf.lite.TFLiteConverter.from_keras_model(modell_mobilenet_v1)
converter_mobilenet_v1.optimizations = [tf.lite.Optimize.DEFAULT]
converter_mobilenet_v1.representative_dataset = representative_dataset_mobilenet_v1
converter_mobilenet_v1.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_mobilenet_v1.inference_input_type = tf.int8
converter_mobilenet_v1.inference_output_type = tf.int8

modell_mobilenet_v1_int8 = converter_mobilenet_v1.convert()

In [ ]:
def representative_dataset_mobilenet_v2():
    for bilder, _ in train_ds.unbatch().batch(1).take(100):
        yield [tf.cast(bilder, tf.float32)]

converter_mobilenet_v2 = tf.lite.TFLiteConverter.from_keras_model(modell_mobilenet_v2)
converter_mobilenet_v2.optimizations = [tf.lite.Optimize.DEFAULT]
converter_mobilenet_v2.representative_dataset = representative_dataset_mobilenet_v2
converter_mobilenet_v2.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_mobilenet_v2.inference_input_type = tf.int8
converter_mobilenet_v2.inference_output_type = tf.int8

modell_mobilenet_v2_int8 = converter_mobilenet_v2.convert()

In [ ]:
def representative_dataset_tinycnn():
    for bilder, _ in train_ds.unbatch().batch(1).take(100):
        yield [tf.cast(bilder, tf.float32)]

converter_tinycnn = tf.lite.TFLiteConverter.from_keras_model(modell_tinycnn)
converter_tinycnn.optimizations = [tf.lite.Optimize.DEFAULT]
converter_tinycnn.representative_dataset = representative_dataset_tinycnn
converter_tinycnn.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_tinycnn.inference_input_type = tf.int8
converter_tinycnn.inference_output_type = tf.int8

modell_tinycnn_int8 = converter_tinycnn.convert()

In [ ]:
with open("/content/pflanzenmodell_mobilenet1_int8.tflite", "wb") as datei:
    datei.write(modell_mobilenet_v1_int8)

In [ ]:
with open("/content/pflanzenmodell_mobilenet2_int8.tflite", "wb") as datei:
    datei.write(modell_mobilenet_v2_int8)

In [ ]:
with open("/content/pflanzenmodell_tinycnn_int8.tflite", "wb") as datei:
    datei.write(modell_tinycnn_int8)

In [ ]:
from google.colab import files
files.download("/content/pflanzenmodell_mobilenet1_int8.tflite")

In [ ]:
from google.colab import files
files.download("/content/pflanzenmodell_mobilenet2_int8.tflite")

In [ ]:
from google.colab import files
files.download("/content/pflanzenmodell_tinycnn_int8.tflite")

In [ ]:
interpreter = tf.lite.Interpreter(model_content=modell_tinycnn_int8)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()[0]

print("Shape:", input_details["shape"])
print("Datentyp:", input_details["dtype"])
print("Quantisierung:", input_details["quantization"])

In [ ]:
interpreter = tf.lite.Interpreter(model_content=modell_mobilenet_v1_int8)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()[0]

print("Shape: ", input_details["shape"])
print("Datentyp:", input_details["dtype"])
print("Quantisierung:", input_details["quantization"])

In [ ]:
interpreter = tf.lite.Interpreter(model_content=modell_mobilenet_v2_int8)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()[0]

print("Shape:", input_details["shape"])
print("Datentyp:", input_details["dtype"])
print("Quantisierung:", input_details["quantization"])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

y_true = []
y_pred = []

for bilder, labels in test_ds:

    vorhersagen = modell_mobilenet_v1.predict(bilder, verbose=0)

    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(vorhersagen, axis=1))

print(classification_report(y_true, y_pred, target_names=["Boden", "Nutzpflanze", "Unkraut"]))

ConfusionMatrixDisplay.from_predictions(
    y_true,
    y_pred,
    labels=[0, 1, 2],
    display_labels=["Boden", "Nutzpflanze", "Unkraut"],
    cmap="Blues"
)

plt.title("MobileNetV1 Float32 Confusion Matrix")
plt.show()

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

interpreter = tf.lite.Interpreter(model_content=modell_mobilenet_v1_int8)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

scale, zero_point = input_details["quantization"]

y_true = []
y_pred = []


for bilder, labels in test_ds:

    for bild, label in zip(bilder, labels):
        bild_int8 = np.round(bild.numpy() / scale + zero_point).astype(np.int8)
        bild_int8 = np.expand_dims(bild_int8,axis=0)
        interpreter.set_tensor(input_details["index"], bild_int8)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details["index"])
        y_true.append(int(label.numpy()))
        y_pred.append(int(np.argmax(output)))

print(classification_report(y_true, y_pred,target_names=["Boden", "Nutzpflanze", "Unkraut"]))

ConfusionMatrixDisplay.from_predictions(
    y_true,
    y_pred,
    labels=[0, 1, 2],
    display_labels=["Boden", "Nutzpflanze", "Unkraut"],
    cmap="Blues"
)

plt.title("MobileNetV1 INT8 Confusion Matrix")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

y_true = []
y_pred = []

for bilder, labels in test_ds:

    vorhersagen = modell_mobilenet_v2.predict(bilder,verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(vorhersagen, axis=1))


print(classification_report(y_true, y_pred, target_names=["Boden", "Nutzpflanze", "Unkraut"]))

ConfusionMatrixDisplay.from_predictions(
    y_true,
    y_pred,
    labels=[0, 1, 2],
    display_labels=["Boden", "Nutzpflanze", "Unkraut"],
    cmap="Blues"
)

plt.title("MobileNetV2 Float32 Confusion Matrix")
plt.show()

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

interpreter = tf.lite.Interpreter(model_content=modell_mobilenet_v2_int8)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

scale, zero_point = input_details["quantization"]

y_true = []
y_pred = []

for bilder, labels in test_ds:

    for bild, label in zip(bilder, labels):
        bild_int8 = np.round(bild.numpy() / scale + zero_point).astype(np.int8)
        bild_int8 = np.expand_dims(bild_int8, axis=0)
        interpreter.set_tensor(input_details["index"], bild_int8)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details["index"])

        y_true.append(int(label.numpy()))
        y_pred.append(int(np.argmax(output)))


print(classification_report(y_true,y_pred, target_names=["Boden", "Nutzpflanze", "Unkraut"]))

ConfusionMatrixDisplay.from_predictions(
    y_true,
    y_pred,
    labels=[0, 1, 2],
    display_labels=["Boden", "Nutzpflanze", "Unkraut"],
    cmap="Blues"
)

plt.title("MobileNetV2 INT8 Confusion Matrix")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

y_true = []
y_pred = []

for bilder, labels in test_ds:

    vorhersagen = modell_tinycnn.predict(bilder,verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(vorhersagen, axis=1))


print(classification_report(y_true,y_pred, target_names=["Boden", "Nutzpflanze", "Unkraut"]))

ConfusionMatrixDisplay.from_predictions(
    y_true,
    y_pred,
    labels=[0, 1, 2],
    display_labels=["Boden", "Nutzpflanze", "Unkraut"],
    cmap="Blues"
)

plt.title("TinyCNN Float32 Confusion Matrix")
plt.show()

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

interpreter = tf.lite.Interpreter(model_content=modell_tinycnn_int8)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]
scale, zero_point = input_details["quantization"]
y_true = []
y_pred = []


for bilder, labels in test_ds:
    for bild, label in zip(bilder, labels):
        bild_int8 = np.round(bild.numpy() / scale + zero_point).astype(np.int8)
        bild_int8 = np.expand_dims(bild_int8,axis=0)
        interpreter.set_tensor(input_details["index"], bild_int8)
        interpreter.invoke()

        output = interpreter.get_tensor(output_details["index"])

        y_true.append(int(label.numpy()))
        y_pred.append(int(np.argmax(output)))

print(classification_report(y_true,y_pred, target_names=["Boden", "Nutzpflanze", "Unkraut"]))

ConfusionMatrixDisplay.from_predictions(
    y_true,
    y_pred,
    labels=[0, 1, 2],
    display_labels=["Boden", "Nutzpflanze", "Unkraut"],
    cmap="Blues"
)

plt.title("TinyCNN INT8 Confusion Matrix")
plt.show()